# BlackPearl Morphix — Stage 3: Does an AI-guided lab find knockout effects with fewer cells?

**The robotics question, tested in simulation.** A lab can only afford a limited number of experiments. Which knockouts should it test next?

**Semi-synthetic virtual lab:**
- Every measured cell is a **real tumour cell** from the Perturb-FISH tissue (its real neighbourhood: T-cell fraction, host fraction, distance to nearest T cell), so cell-to-cell variation and spatial structure are real.
- On top, we **plant knockout effects** at the sizes Stage 2 measured: 8 of 30 knockouts shift one neighbourhood feature by 0.2–0.6 SD, one by 1.0 SD; the rest do nothing. Because we planted them, we know the right answer.
- One **experiment** = measure 50 cells carrying one knockout.

**Strategies (same budget, same rules for calling a hit):**
- **Round robin:** test every knockout in turn (the standard screen).
- **Random:** pick knockouts at random.
- **Active (Morphix loop):** after one screening experiment per knockout, always test the most promising knockout not yet confirmed, and stop spending on knockouts that already look confidently null.

**Calling a hit** (identical for all): posterior probability that |effect| > 0.1 SD is above 99%, **and** the knockout was measured in at least 150 cells (confirmed by repeats). This rule and the active strategy were designed on mock tissue *before* touching this data.

**Second question, experimental design:** the same screen run as a **mosaic** (each experiment's cells come from one contiguous patch, like a pooled clone in the Perturb-FISH tumour) vs **randomised** placement.

CPU runtime is fine (no GPU). ~5–10 minutes. Upload `morphix_pfish_cells (1).csv.gz` when asked. Copy the **SUMMARY** at the end and paste it back.

*Scope: this validates the experiment-selection logic under realistic noise. It does not simulate how real knockouts change tissue; the planted effects are additive shifts.*

In [ ]:
import os, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
CFG = dict(n_genes=30, budget_experiments=300, cells_per_experiment=50, n_worlds=100, n_worlds_hard=50,
           delta=0.1, threshold=0.99, min_cells_to_confirm=150, n_focal=40000, seed=0)
os.makedirs('results_s3', exist_ok=True)
S = {'config': CFG}

In [ ]:
# ---------- STAGE 3: simulated closed-loop experiment design ----------
import os, glob, json
import numpy as np, pandas as pd
from scipy.spatial import cKDTree
from scipy.stats import norm

FEATS = ['frac_tcell', 'frac_host', 'dist_to_tcell']


def load_cells():
    cands = sorted(glob.glob('morphix_pfish_cells*.csv.gz'), key=os.path.getmtime)
    if not cands:
        from google.colab import files
        print('Upload morphix_pfish_cells (1).csv.gz (from the successful Step 2 run):')
        up = files.upload()
        cands = [n for n in up if n.startswith('morphix_pfish_cells')]
    cells = pd.read_csv(cands[-1], index_col='cell_id')
    if not (cells.cell_class == 'tumour').any():
        raise ValueError('This is the old file from the failed run; upload the newer one.')
    return cells


def real_neighbourhood_features(cells, n_focal=40000, k=63, seed=0):
    """Neighbourhood features of real tumour-like cells (same definitions as Stage 2). Standardised, so every
    simulated effect is in SD units of real cell-to-cell variation."""
    rng = np.random.default_rng(seed)
    tc = np.log10(cells.total_counts + 1)
    thr = np.quantile(tc[cells.cell_class == 'tumour'], 0.05)
    cls = np.where(cells.cell_class == 'tcell', 1, np.where((cells.cell_class == 'tumour') | (tc >= thr), 0, 2))
    XY = cells[['x', 'y']].values
    tree = cKDTree(XY)
    unit = float(np.median(tree.query(XY, k=2)[0][:, 1]))
    focal = np.flatnonzero(cls == 0)
    focal = rng.choice(focal, min(n_focal, len(focal)), replace=False)
    d, nb = tree.query(XY[focal], k=k + 1)
    d, nb = d[:, 1:] / unit, nb[:, 1:]
    c = cls[nb]
    dt = np.where(c == 1, d, np.inf).min(1)
    dt = np.where(np.isfinite(dt), dt, d.max(1))
    F = np.c_[(c == 1).mean(1), (c == 2).mean(1), dt]
    mu, sd = F.mean(0), F.std(0)
    return (F - mu) / sd, XY[focal], unit


class VirtualLab:
    """A lab that can run experiments. Experiment = measure n cells carrying knockout g; each cell's readout is a
    REAL tumour cell's neighbourhood (real variance and spatial structure) plus the knockout's TRUE effect.
    design='randomised': cells drawn at random positions (arrayed, randomised layout).
    design='mosaic': cells drawn from one contiguous patch (a clone), like the pooled Perturb-FISH screen, so
    local tissue differences leak into every measurement."""
    def __init__(self, F, XY, effects, design, rng, n_per_exp=50):
        self.F, self.XY, self.effects, self.design, self.rng, self.n = F, XY, effects, design, rng, n_per_exp
        self.tree = cKDTree(XY) if design == 'mosaic' else None

    def run(self, g):
        if self.design == 'randomised':
            idx = self.rng.integers(0, len(self.F), self.n)      # random cells anywhere in the tissue
        else:
            centre = self.XY[self.rng.integers(len(self.XY))]
            _, idx = self.tree.query(centre, k=self.n)
        return self.F[idx] + self.effects[g]          # [n, n_feats]


def make_world(n_genes, rng, sizes=(0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.6, 1.0)):
    """True effects: most knockouts do nothing; a few shift ONE neighbourhood feature by a measured-scale amount
    (Stage 2 found ~0.2-0.6 SD effects and one ~1 SD effect, in about a quarter of knockouts)."""
    E = np.zeros((n_genes, len(FEATS)))
    hits = rng.choice(n_genes, len(sizes), replace=False)
    for g, s in zip(hits, sizes):
        E[g, rng.integers(len(FEATS))] = s * rng.choice([-1, 1])
    return E, set(hits.tolist())


class Posterior:
    """Per-knockout, per-feature Gaussian posterior on the effect (prior N(0, tau^2), unit cell-level noise).
    Bayesian calls stay valid when the next experiment depends on earlier results."""
    def __init__(self, n_genes, n_feats, tau=0.5):
        self.s = np.zeros((n_genes, n_feats)); self.n = np.zeros(n_genes); self.tau2 = tau ** 2

    def add(self, g, Y):
        self.s[g] += Y.sum(0); self.n[g] += len(Y)

    def stats(self):
        prec = 1 / self.tau2 + self.n[:, None]
        return self.s / prec, 1 / prec                       # posterior mean, variance

    def p_real(self, delta):
        m, v = self.stats(); sd = np.sqrt(v)
        return norm.sf((delta - m) / sd) + norm.cdf((-delta - m) / sd)   # P(|effect| > delta)


def confirmed(post, delta, threshold, min_cells):
    """A knockout counts as a hit only if P(|effect| > delta) > threshold AND it has been measured in at least
    min_cells cells (i.e. confirmed by repeat experiments). Same rule for every strategy."""
    return (post.p_real(delta).max(1) > threshold) & (post.n >= min_cells)


def choose_next(strategy, post, step, n_genes, rng, delta, threshold, min_cells):
    if strategy == 'round_robin' or step < n_genes:          # everyone starts with one screening experiment
        return step % n_genes
    if strategy == 'random':
        return int(rng.integers(n_genes))
    if strategy == 'active':
        # Test the most promising knockout that is not yet a confirmed hit; ignore knockouts that already look
        # confidently null (P(real) < 5%). This spends experiments where they can still change the answer.
        p = post.p_real(delta).max(1)
        open_ = ~confirmed(post, delta, threshold, min_cells)
        cand = np.flatnonzero(open_ & (p > 0.05))
        if len(cand) == 0:
            cand = np.flatnonzero(open_)
        return int(cand[p[cand].argmax()]) if len(cand) else int(rng.integers(n_genes))
    raise ValueError(strategy)


def calls(post, delta, threshold, min_cells):
    return set(np.flatnonzero(confirmed(post, delta, threshold, min_cells)).tolist())


def run_campaign(F, XY, strategy, design, n_genes, budget, rng, n_per_exp=50, delta=0.1, threshold=0.99,
                 min_cells=150, checkpoints=None, size_scale=1.0, world=0):
    E, truth = make_world(n_genes, rng)
    E = E * size_scale
    lab = VirtualLab(F, XY, E, design, rng, n_per_exp)
    post = Posterior(n_genes, F.shape[1])
    out = []
    for step in range(budget):
        g = choose_next(strategy, post, step, n_genes, rng, delta, threshold, min_cells)
        post.add(g, lab.run(g))
        if checkpoints is None or (step + 1) in checkpoints:
            c = calls(post, delta, threshold, min_cells)
            tp = len(c & truth)
            m, _ = post.stats()
            err = np.mean([np.abs(m[g] - E[g]).max() for g in truth])
            out.append(dict(experiments=step + 1, cells=(step + 1) * n_per_exp, recall=tp / len(truth),
                            false_discoveries=len(c - truth), fdr=(len(c - truth) / max(len(c), 1)),
                            effect_error=err, strategy=strategy, design=design, world=world,
                            size_scale=size_scale))
    return out

## 1. Real tissue as the virtual lab's cell pool

In [ ]:
cells = load_cells()
F, XY, UNIT = real_neighbourhood_features(cells, CFG['n_focal'], seed=CFG['seed'])
print(f'{len(F)} real tumour-like cells; features (standardised): {FEATS}')

# Why mosaic designs mislead: spread of 50-cell averages from random cells vs from one contiguous patch
rng = np.random.default_rng(1); tree = cKDTree(XY)
rand_means = np.array([F[rng.integers(0, len(F), 50)].mean(0) for _ in range(2000)])
patch_means = np.array([F[tree.query(XY[rng.integers(len(XY))], k=50)[1]].mean(0) for _ in range(2000)])
spread = pd.DataFrame({'random 50 cells': rand_means.std(0), 'one 50-cell patch': patch_means.std(0),
                       'expected if independent (1/sqrt(50))': [1 / np.sqrt(50)] * 3}, index=FEATS)
print('\nSpread (SD units) of a 50-cell experiment average with NO knockout effect:')
print(spread.round(3))
S['no_effect_spread_of_50cell_means'] = spread.round(4).to_dict()

## 2. Main experiment: AI-guided vs standard screening (randomised design)

In [ ]:
CP = list(range(5, CFG['budget_experiments'] + 1, 5))
rows, t0 = [], time.time()
for w in range(CFG['n_worlds']):
    for strat in ['round_robin', 'random', 'active']:
        rows += run_campaign(F, XY, strat, 'randomised', CFG['n_genes'], CFG['budget_experiments'],
                             np.random.default_rng(10_000 + w), CFG['cells_per_experiment'], CFG['delta'],
                             CFG['threshold'], CFG['min_cells_to_confirm'], CP, 1.0, w)
main = pd.DataFrame(rows)
print(f'{CFG["n_worlds"]} simulated worlds x 3 strategies in {time.time() - t0:.0f}s')

def cells_to(df, target):
    # per world: first cell count at which recall >= target (inf if never)
    out = {}
    for (s, w), d in df.groupby(['strategy', 'world']):
        hit = d[d.recall >= target]
        out.setdefault(s, []).append(hit.cells.min() if len(hit) else np.inf)
    return {s: np.array(v) for s, v in out.items()}

summ = {}
for target in [0.8, 0.9]:
    ct = cells_to(main, target)
    for s, v in ct.items():
        summ[f'{s}: cells to {int(target * 100)}% recall (median)'] = float(np.median(v))
        summ[f'{s}: worlds reaching {int(target * 100)}% within budget'] = f'{np.isfinite(v).sum()}/{len(v)}'
fd = main[main.experiments == CFG['budget_experiments']].groupby('strategy')[['recall', 'false_discoveries', 'effect_error']].mean()
for s in fd.index:
    summ[f'{s}: recall at full budget'] = round(float(fd.loc[s, 'recall']), 3)
    summ[f'{s}: false discoveries per campaign (full budget)'] = round(float(fd.loc[s, 'false_discoveries']), 3)
    summ[f'{s}: mean abs error of true effects (SD)'] = round(float(fd.loc[s, 'effect_error']), 3)
for k, v in summ.items(): print(f'{k:60s} {v}')
r80 = summ['round_robin: cells to 80% recall (median)'] / summ['active: cells to 80% recall (median)']
print(f'\n==> HEADLINE: active needs {r80:.1f}x fewer cells than round robin to find 80% of real effects (median over worlds)')
a90, rr90 = summ['active: cells to 90% recall (median)'], summ['round_robin: cells to 90% recall (median)']
if np.isfinite(rr90):
    print(f'==> 90% of effects: active {a90:.0f} cells vs round robin {rr90:.0f} cells ({rr90 / a90:.1f}x)')
else:
    print(f'==> 90% of effects: active needs {a90:.0f} cells; round robin reaches 90% in only '
          f'{summ["round_robin: worlds reaching 90% within budget"]} worlds even with {CFG["budget_experiments"] * CFG["cells_per_experiment"]} cells')
S['main'] = summ; S['fold_fewer_cells_to_80pct_vs_round_robin'] = round(float(r80), 2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for s, col in [('round_robin', 'tab:grey'), ('random', 'tab:orange'), ('active', 'tab:blue')]:
    d = main[main.strategy == s].groupby('cells')
    m, lo, hi = d.recall.mean(), d.recall.quantile(0.1), d.recall.quantile(0.9)
    ax[0].plot(m.index, m, c=col, lw=2, label=s); ax[0].fill_between(m.index, lo, hi, color=col, alpha=0.15)
    ax[1].plot(m.index, d.false_discoveries.mean(), c=col, lw=2, label=s)
ax[0].set_xlabel('cells measured'); ax[0].set_ylabel('fraction of real effects found'); ax[0].axhline(0.9, ls=':', c='k')
ax[0].set_title('Finding real knockout effects (band: 10-90% of worlds)'); ax[0].legend()
ax[1].set_xlabel('cells measured'); ax[1].set_ylabel('false hits per campaign'); ax[1].set_title('False discoveries'); ax[1].legend()
plt.tight_layout(); plt.savefig('results_s3/recall_vs_cells.png', dpi=150); plt.show()

## 3. Harder world: effects half as large

In [ ]:
rows = []
for w in range(CFG['n_worlds_hard']):
    for strat in ['round_robin', 'active']:
        rows += run_campaign(F, XY, strat, 'randomised', CFG['n_genes'], CFG['budget_experiments'],
                             np.random.default_rng(20_000 + w), CFG['cells_per_experiment'], CFG['delta'],
                             CFG['threshold'], CFG['min_cells_to_confirm'], CP, 0.5, w)
hard = pd.DataFrame(rows)
end = hard[hard.experiments == CFG['budget_experiments']].groupby('strategy')[['recall', 'false_discoveries']].mean()
print('Effects x0.5 (0.1-0.3 SD, one 0.5 SD). At full budget (15,000 cells):'); print(end.round(3))
ct = cells_to(hard, 0.5)
for s, v in ct.items():
    print(f'{s}: cells to 50% recall (median) = {np.median(v)}, worlds reaching it: {np.isfinite(v).sum()}/{len(v)}')
S['hard_world_full_budget'] = end.round(3).to_dict()
S['hard_world_cells_to_50pct'] = {s: float(np.median(v)) for s, v in ct.items()}

## 4. Experimental design: mosaic (pooled clones) vs randomised placement

In [ ]:
rows = []
for w in range(50):
    for design in ['randomised', 'mosaic']:
        rows += run_campaign(F, XY, 'round_robin', design, CFG['n_genes'], CFG['budget_experiments'],
                             np.random.default_rng(30_000 + w), CFG['cells_per_experiment'], CFG['delta'],
                             CFG['threshold'], CFG['min_cells_to_confirm'], CP, 1.0, w)
des = pd.DataFrame(rows)
end = des[des.experiments == CFG['budget_experiments']].groupby('design')[['recall', 'false_discoveries', 'fdr']].mean()
print('Same screen, same analysis, different physical layout (full budget):'); print(end.round(3))
S['design_comparison_full_budget'] = end.round(3).to_dict()
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
end['recall'].plot.bar(ax=ax[0], color=['tab:red', 'tab:green']); ax[0].set_title('Real effects found'); ax[0].set_ylim(0, 1.05)
end['false_discoveries'].plot.bar(ax=ax[1], color=['tab:red', 'tab:green']); ax[1].set_title('False hits per campaign (of 22 null knockouts)')
for a in ax: a.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.savefig('results_s3/design_comparison.png', dpi=150); plt.show()

In [ ]:
main.to_csv('results_s3/main_trajectories.csv', index=False)
!zip -qr morphix_stage3_results.zip results_s3
try:
    from google.colab import files; files.download('morphix_stage3_results.zip')
except Exception:
    pass

## How to read it
- **Cells to 90% recall:** how much lab work each strategy needs to find nearly all real effects. The fold difference is the headline.
- **False hits:** must stay near zero for every strategy, otherwise the speed-up is not worth it.
- **Mosaic vs randomised:** if the mosaic layout produces many false hits, that is the location confound from Stage 2, now shown in a setting where we know the truth, and the reason a robotic lab should use randomised, arrayed layouts.

## SUMMARY — copy everything printed below and paste it back

In [ ]:
print('===== SUMMARY START =====')
print(json.dumps(S, indent=1, default=str)[:15000])
print('===== SUMMARY END =====')